# 5.13 資料或模型變大，哪個值得花預算？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**大模型看更多資料也訓練更久，能知道進步來自哪裡嗎？

先畫出兩種模型容量×兩種資料量的四格設計。再決定要固定步數、tokens、FLOPs或時間；這些不是同一種公平。

**把直覺寫成數學：**密集Linear成本大致隨矩陣乘法維度乘積增加；不能用四個tiny點外推大型scaling law。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

for width in (8, 16):
    parameters = TinyLM(ModelConfig(width=width)).description()["parameters"]
    for records in (16, 64):
        print(
            {
                "width": width,
                "records": records,
                "parameters": parameters,
                "steps": 100,
                "quality": "待讀者執行訓練後填入",
            }
        )

**如何讀結果：**這是實驗計畫，不是測量的品質表；保存原始JSONL與實際有效tokens。

**只改一件事：**只把四格改成相同token預算，再算所需步數。
